# PRIM — Facility Pooling (fac=1, amb=0)

Find where T1 mortality > 0.


In [ ]:
import sys
from pathlib import Path
_root = Path(".").resolve()
while not (_root / "paths.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RESULTS_PATH = _root / "data_cluster/saltelli_with_wrong_pooling_columns/results.csv"
df = pd.read_csv(RESULTS_PATH)
print(f"Loaded {len(df)} runs")

# ── PRIM CONFIGURATION ────────────────────────────────────────
PRIM_OUTCOME = "death_rate_T1_BCAS"  # <-- change to any outcome column
PRIM_THRESHOLD = 0.0                 # <-- mortality > this is "interesting"
PRIM_DIRECTION = "above"             # <-- "above" = outcome > threshold, "below" = outcome < threshold
# ─────────────────────────────────────────────────────────────

UNCERTAINTIES = ["ccp_r1_same_bn_min", "ccp_r1_inter_bn_min", "bcas_rate", "waves_per_day", "wave_duration_min"]

sub = df[(df["facility_eligibility"] == 1) & (df["ambulance_eligibility"] == 0)].reset_index(drop=True)
print(f"Policy: Facility Pooling (fac=1, amb=0), {len(sub)} runs")
print(f"T1 mortality range: {sub['death_rate_T1_BCAS'].min():.4f} - {sub['death_rate_T1_BCAS'].max():.4f}")
print(f"Nonzero: {(sub['death_rate_T1_BCAS'] > 0).sum()} / {len(sub)} ({(sub['death_rate_T1_BCAS'] > 0).mean():.1%})")


# PRIM — Where is T1 mortality ZERO?


In [ ]:
from ema_workbench.analysis import prim

x = sub[UNCERTAINTIES].astype(float)
if PRIM_DIRECTION == "above":
    y = sub[PRIM_OUTCOME].values > PRIM_THRESHOLD
    label = f"{PRIM_OUTCOME} > {PRIM_THRESHOLD}"
else:
    y = sub[PRIM_OUTCOME].values < PRIM_THRESHOLD
    label = f"{PRIM_OUTCOME} < {PRIM_THRESHOLD}"

print(f"PRIM target: {label}")
print(f"Cases of interest: {y.sum()} / {len(y)} ({y.mean():.1%})")

p = prim.Prim(x, y, threshold=0.5)
box = p.find_box()
box.show_tradeoff()
print(f"
Selected box ({label}):")
box.inspect(style="graph")
box.inspect()
